In [1]:
import os
import json
from transformers import pipeline

W0226 11:10:39.765000 17936 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# Get the path to the datafile folder
current_dir = os.path.dirname(os.path.abspath("*"))
data_folder = os.path.join(current_dir, "..", "datafile")

all_questions = []

# Loop through all JSON files
for filename in os.listdir(data_folder):
    if filename.endswith(".json"):
        file_path = os.path.join(data_folder, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

            # Access freshman_sophomore_test
            test_data = data.get("freshman_sophomore_test", {})

            for q_key, q_value in test_data.items():
                question_text = q_value.get("question")
                answer_text = q_value.get("answer")

                all_questions.append({
                    "year": filename.replace(".json", ""),
                    "question_id": q_key,
                    "question": question_text,
                    "answer": answer_text
                })

In [3]:
all_questions

[{'year': '2022',
  'question_id': 'question_1',
  'question': 'Determine which of the following five statement(s) is sufficient to deduct that x > y. A) x + 1 = y  B) x + 2.2 = y  C) x - 1.3 = y  D) xy > 0  E) xy < 0',
  'answer': 'C'},
 {'year': '2022',
  'question_id': 'question_2',
  'question': 'Jadyn just completed a 4-day trip. On the first day she drove 531 miles; on the second day, 615 miles; on the third day, 704 miles; on the fourth day, k miles. If Jadyn drove an average of 622.5 miles per day for the 4-day trip, determine the value of k.',
  'answer': '640'},
 {'year': '2022',
  'question_id': 'question_3',
  'question': 'A rectangle is half as wide as it is long and has a perimeter of x units. The numeric area of this rectangle can be written as the expression kx². Determine the value of k. Express your answer as an integer or as a common or improper fraction reduced to lowest terms.',
  'answer': '1/18'},
 {'year': '2022',
  'question_id': 'question_4',
  'question': 'A 

In [8]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="your_groq_api_key",  # free at console.groq.com
)
# Change model to:
model="llama-3.1-8b-instant"  # or "deepseek-r1-distill-llama-70b" for reasoning

In [15]:
import json
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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",
)

MODEL = "openai/gpt-oss-120b"

def call_model(messages):
    completion = client.chat.completions.create(
        model=MODEL,  # ← fixed: was still "deepseek-ai/DeepSeek-R1:together"
        messages=messages
    )
    content = completion.choices[0].message.content

    if "</think>" in content:
        content = content.split("</think>")[-1].strip()

    return content

results = []

for q in all_questions:
    print(f"\n{'='*60}")
    print(f"[{q['year']} - {q['question_id']}]")
    print(f"Question: {q['question']}")
    print(f"Correct Answer: {q['answer']}")

    solver_messages = [
        {
            "role": "system",
            "content": (
                "You are a precise math competition solver. "
                "Solve the problem step by step, then end with:\n"
                "FINAL ANSWER: "
            )
        },
        {"role": "user", "content": q["question"]}
    ]
    solver_reply = call_model(solver_messages)
    print(f"\n[Solver]\n{solver_reply}")

    judge_messages = [
        {
            "role": "system",
            "content": (
                "You are a strict math answer evaluator.\n"
                "You will be given a question, the correct answer, and a student's response.\n"
                "Determine if the student's final answer is mathematically equivalent to the correct answer.\n\n"
                "Respond with EXACTLY one of the following and nothing else:\n"
                "VERDICT: CORRECT\n"
                "VERDICT: INCORRECT"
            )
        },
        {
            "role": "user",
            "content": (
                f"Question: {q['question']}\n\n"
                f"Correct Answer: {q['answer']}\n\n"
                f"Student's Response:\n{solver_reply}"
            )
        }
    ]
    judge_reply = call_model(judge_messages)
    print(f"\n[Judge]\n{judge_reply}")

    results.append({
        "year": q["year"],
        "question_id": q["question_id"],
        "question": q["question"],
        "correct_answer": q["answer"],
        "solver_response": solver_reply,
        "judge_response": judge_reply
    })

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n\nDone. {len(results)} questions evaluated. Results saved to results.json")


[2022 - question_1]
Question: Determine which of the following five statement(s) is sufficient to deduct that x > y. A) x + 1 = y  B) x + 2.2 = y  C) x - 1.3 = y  D) xy > 0  E) xy < 0
Correct Answer: C

[Solver]
**Step‑by‑step analysis**

We consider real numbers \(x\) and \(y\).  
We must decide, for each statement, whether the statement **alone** guarantees that \(x>y\).

---

### A) \(x+1 = y\)

\[
x+1 = y \;\Longrightarrow\; x = y-1.
\]

Since we subtract a positive number (1) from \(y\), we have  

\[
x = y-1 < y.
\]

Thus the statement forces \(x<y\), not \(x>y\).  
**Not sufficient.**

---

### B) \(x+2.2 = y\)

\[
x+2.2 = y \;\Longrightarrow\; x = y-2.2.
\]

Again we subtract a positive quantity (2.2) from \(y\), so  

\[
x = y-2.2 < y.
\]

The inequality is opposite of what we need.  
**Not sufficient.**

---

### C) \(x-1.3 = y\)

\[
x-1.3 = y \;\Longrightarrow\; x = y+1.3.
\]

Here we **add** a positive number (1.3) to \(y\). Therefore  

\[
x = y+1.3 > y.
\]

The statement

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01j62tj9djf32943edf1z99pq1` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 200000, Requested 544. Please try again in 3m55.008s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}